# **Preprocessing Data** - **Using Random Forest Regressor to Estimate Density-Porosity Well Log Readings**

#### **Importing some important libraries for ML**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_squared_error

- pandas and numpy essential to manipulate dataframe
- matplotlib and seaborn for data manipulation
- sklearn for machine learning libraries
- Include GridSearch CV for hypertuning parameters

#### **Extracting dataframe and execute Pearson Correlation Matrix**

In [ ]:
#Extracting dataframe
raw = pd.read_csv("filenamehere.csv")
#checking the collinearity through correlation heatmap
features = ["Depth", "RxoRt", "RLL3", "SP", "RILD", "MN", "MI", "MCAL", "DCAL", "CNLS", "GR"]
sns.heatmap(data=raw[features].corr(),annot=True)
plt.show()

![Correlation Heatmap](Correlation_heatmap_project1.png)

- using pd.read_csv() to import in as pd dataframe
- using seaborn heatmap to find multicollinearity 


#### **Addressing data dropping base on petrophysics concept**

We saw a  multicollinearity and possible data leakage between the features, we need to drop RHOB,RHOC and MCAL MI
1) Density-Porosity Log (DPOR) are mathematically derived from RHOB,RHOP. As the algorithm train, it will start to find really easy pattern in the data.
    Lets just say, its become more straightfoward and just soon become data leakage. It happens when the weight of synthesized equation from 
    learning RHOC and RHOB data has been adjusted perfectly with DPOR.
    
2) MCAL and DCAL, MN and MI found to have a Pearson correlation coefficient, r almost equals to one (0.98 - 0.90). Meaning that, if we put all the data into
    our machine (especially Random Forest, a tree-based decision algo),it will dilute our feature importance. More of the feature is redundant and it will actually hide the true potential of 
    our machine's predictive power.

3) After a little research, I figured out that DCAL and DPOR are actually taken from the same time in the wellbore (Using density tool string), while MCAL are taken on a separate time. As a petroleum engineer
    student, I need to know that wellbore is a very dynamic system as the wellbore condition may vary, thus the MCAL may actually deviate from the time of recorded DPOR.
    Thus, DCAL is a better representative for our data to estimate DPOR.

4) For MN and MI, its fundamental is still complicated for me to choose as both of them are relying on each other to identify if the rock surrounding the wellbore are 
    permeable or not. I retain MN as it use to identify the rock behavior, while MI use to check the mudcake surrounding the wellbore.

#### **Defining variables and checking the Dataframe**

In [ ]:
filtered_features = ["Depth","RxoRt", "RLL3", "SP", "RILD", "MN","DCAL", "CNLS", "GR"]
X = raw[filtered_features]
Y = raw["DPOR"]
print(X.head())
print(Y.head())

#### **Data Splitting and Standardisation**

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,Y,test_size=0.25,random_state=0)
#Scaling the features
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)


- Using sklearn's train_test_split()
- Standard scaler use to avoid any numerical bias.
- The data is pretty straightfoward. My assumption is that, if huge chunk of data is not present and dirty, I will try to predict those value first (e.g. predicting resisitvity first before using those predicted data to predict DPOR, which will alter the way of data splitting) and need to use other data splitting first technique rather than sklearn's function.
- So, the test data will be only 25% of the total datasets, the rest is for machine's training.